# Player Classifier GPU Verification Test

このNotebookは、Google Colab上でGPUを使ってプレイヤー分類器の検証テストを実行します。

## 機能
- YOLOv11-poseによる人物姿勢検出
- TableDetectorによる卓球台検出
- PlayerClassifierによるプレイヤー特定
- GPU加速による高速処理
- 結果の可視化と動画出力

## 準備
1. ランタイム > ランタイムのタイプを変更 > **GPU** を選択
2. 以下のファイルをアップロード:
   - `src/detection/` フォルダの全ファイル（4つ）
     - `data_classes.py`
     - `table_detector.py`
     - `yolopose_tracker.py`
     - `player_classifier.py`
   - `models/proto_type02_table_detection_models/best.pt`（卓球台検出モデル）
   - テスト用動画ファイル

## 1. 環境セットアップ

In [ ]:
# GPU確認
!nvidia-smi

In [ ]:
# 必要なパッケージのインストール
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.26.4
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# ディレクトリ構造を作成
import os
os.makedirs('src/detection', exist_ok=True)
os.makedirs('models/proto_type02_table_detection_models', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("✓ ディレクトリ構造を作成しました")

## 2. ファイルのアップロード

In [ ]:
# テスト用動画をアップロード
from google.colab import files
print("テスト用動画ファイルをアップロードしてください...")
uploaded = files.upload()

# アップロードされた最初の動画ファイルを使用
video_files = [f for f in uploaded.keys() if f.lower().endswith(('.mp4', '.mov', '.avi'))]
if video_files:
    input_video = video_files[0]
    print(f"✓ 動画ファイル '{input_video}' をアップロードしました")
else:
    print("⚠ 動画ファイルが見つかりません")
    input_video = None

## 3. モジュールのインポート

In [ ]:
# 必要なモジュールをインポート
import cv2
import numpy as np
import sys
from pathlib import Path
from typing import Optional, List, Set
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode

# プロジェクトのルートディレクトリをパスに追加
sys.path.insert(0, '.')

# プロジェクトモジュールをインポート
from src.detection.table_detector import TableDetector
from src.detection.yolopose_tracker import YOLOPose_Tracker
from src.detection.player_classifier import PlayerClassifier
from src.detection.data_classes import TableInfo, PersonTrack

print("✓ モジュールのインポートが完了しました")

## 4. 可視化クラスの定義

In [ ]:
# 可視化クラスの定義
class PlayerClassifierVisualizer:
    """プレイヤー分類結果を可視化するクラス"""
    
    def __init__(self, table_detector, pose_tracker, player_classifier):
        self.table_detector = table_detector
        self.pose_tracker = pose_tracker
        self.player_classifier = player_classifier
    
    def draw_results(self, frame, table_info, persons, player_ids):
        """検出結果を描画"""
        output = frame.copy()
        
        # 卓球台を描画
        if table_info:
            x1, y1, x2, y2 = table_info.bbox
            cv2.rectangle(output, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 255), 2)
            cv2.putText(output, "Table", (int(x1), int(y1) - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        else:
            # 卓球台が検出できていない場合の警告表示
            cv2.putText(output, "WARNING: Table Not Detected", (20, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
        
        # スケルトン接続定義
        skeleton_connections = [
            (0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 6),
            (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16)
        ]
        
        # 人物を描画
        for person in persons:
            is_player = person.track_id in player_ids
            color = (0, 255, 0) if is_player else (0, 0, 255)
            label = "PLAYER" if is_player else "Other"
            
            # バウンディングボックス
            x1, y1, x2, y2 = person.bbox
            thickness = 3 if is_player else 2
            cv2.rectangle(output, (x1, y1), (x2, y2), color, thickness)
            
            # ラベル
            text = f"{label} ID:{person.track_id}"
            cv2.putText(output, text, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            # スケルトン
            for connection in skeleton_connections:
                kp1_idx, kp2_idx = connection
                kp1 = person.keypoints[kp1_idx]
                kp2 = person.keypoints[kp2_idx]
                
                if kp1[2] > 0.5 and kp2[2] > 0.5:
                    pt1 = (int(kp1[0]), int(kp1[1]))
                    pt2 = (int(kp2[0]), int(kp2[1]))
                    cv2.line(output, pt1, pt2, color, 2)
            
            # キーポイント
            for kp in person.keypoints:
                if kp[2] > 0.5:
                    pt = (int(kp[0]), int(kp[1]))
                    cv2.circle(output, pt, 3, color, -1)
        
        return output
    
    def draw_candidate_info(self, frame, player_ids):
        """候補者情報を描画"""
        output = frame.copy()
        info_x = frame.shape[1] - 400
        info_y = 30
        line_height = 25
        
        cv2.putText(output, "=== Player Candidates ===", (info_x, info_y),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        y_offset = info_y + line_height
        
        # 候補情報を取得してスコア順にソート
        candidates = []
        for track_id, candidate in self.player_classifier.candidates.items():
            if candidate.total_frames >= self.player_classifier.min_tracking_frames:
                score = self.player_classifier._calculate_player_score(candidate)
                candidates.append((track_id, candidate, score))
        
        candidates.sort(key=lambda x: x[2], reverse=True)
        
        # 上位候補を表示
        for track_id, candidate, score in candidates[:5]:
            is_player = track_id in player_ids
            color = (0, 255, 0) if is_player else (200, 200, 200)
            
            text = f"ID:{track_id} {'[PLAYER]' if is_player else ''}"
            cv2.putText(output, text, (info_x, y_offset),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            y_offset += line_height
            
            text = f"  Score: {score:.3f}"
            cv2.putText(output, text, (info_x, y_offset),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
            
            text = f"  Frames: {candidate.total_frames}, Move: {candidate.total_movement:.1f}"
            cv2.putText(output, text, (info_x, y_offset),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
            
            text = f"  Near table: {candidate.near_table_ratio:.1%}"
            cv2.putText(output, text, (info_x, y_offset),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += line_height
        
        return output

print("✓ Visualizerクラスの定義が完了しました")

## 5. テストパラメータの設定

In [ ]:
# テストパラメータ設定
INPUT_VIDEO = input_video if input_video else 'sample_video.mp4'
OUTPUT_VIDEO = 'output/player_classification_result.mp4'
TABLE_MODEL = 'models/proto_type02_table_detection_models/best.pt'
POSE_MODEL = 'models/yolo11n-pose.pt'

# 処理パラメータ（必要に応じて変更してください）
FPS = 30.0              # 処理FPS（GPU利用時は高FPS推奨）
MAX_PLAYERS = 4         # 最大プレイヤー数
MIN_PLAYER_SCORE = 0.3  # プレイヤー判定の最小スコア閾値（0.0-1.0）

print("テストパラメータ:")
print(f"  入力動画: {INPUT_VIDEO}")
print(f"  出力動画: {OUTPUT_VIDEO}")
print(f"  処理FPS: {FPS}")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE}")

## 6. テストの実行

In [ ]:
# メイン処理
def run_player_classification_test():
    """プレイヤー分類テストを実行"""
    
    # 動画を開く
    print(f"\n動画ファイルを開いています: {INPUT_VIDEO}...")
    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        print("エラー: 動画ファイルを開けませんでした")
        return
    
    # フレーム情報を取得
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n入力情報:")
    print(f"  解像度: {width}x{height}")
    print(f"  FPS: {video_fps:.2f}")
    print(f"  総フレーム数: {total_frames}")
    print(f"  処理FPS: {FPS:.2f}\n")
    
    # フレーム間隔を計算
    frame_interval = max(1, int(video_fps / FPS))
    
    # コンポーネントを初期化
    print("コンポーネントを初期化しています...")
    table_detector = TableDetector(yolo_model_path=TABLE_MODEL)
    pose_tracker = YOLOPose_Tracker(model_path=POSE_MODEL)
    player_classifier = PlayerClassifier(
        max_players=MAX_PLAYERS,
        min_player_score=MIN_PLAYER_SCORE
    )
    visualizer = PlayerClassifierVisualizer(
        table_detector, pose_tracker, player_classifier
    )
    
    print(f"プレイヤー分類設定:")
    print(f"  最大プレイヤー数: {MAX_PLAYERS}")
    print(f"  最小スコア閾値: {MIN_PLAYER_SCORE:.2f}\n")
    
    # 出力ビデオの準備
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, video_fps, (width, height))
    
    # 卓球台を検出
    print("卓球台を検出中...")
    table_info = None
    max_detection_attempts = 100
    
    for attempt in range(max_detection_attempts):
        ret, frame = cap.read()
        if not ret:
            print("エラー: 動画の終端に達しました")
            cap.release()
            video_writer.release()
            return
        
        table_info = table_detector.detect_table_from_frame(
            frame, frame_idx=attempt, force_detect=True
        )
        
        if table_info is not None:
            print(f"✓ 卓球台を検出しました（フレーム {attempt + 1}、信頼度: {table_info.confidence:.2f}）\n")
            break
    
    if table_info is None:
        print(f"エラー: 卓球台を検出できませんでした")
        cap.release()
        video_writer.release()
        return
    
    # 動画を最初に戻す
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    # フレーム処理
    frame_count = 0
    processed_count = 0
    player_ids = set()
    last_persons = []
    
    print("処理開始...\n")
    
    # プログレスバー付きで処理
    pbar = tqdm(total=total_frames, desc="Processing")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            pbar.update(1)
            
            # 指定FPSで処理
            if frame_count % frame_interval != 0:
                if processed_count > 0:
                    display_frame = visualizer.draw_results(
                        frame, table_info, last_persons, player_ids
                    )
                    display_frame = visualizer.draw_candidate_info(
                        display_frame, player_ids
                    )
                else:
                    display_frame = frame.copy()
                
                video_writer.write(display_frame)
                continue
            
            processed_count += 1
            
            # 人物を検出・追跡
            if table_info:
                persons = pose_tracker.track_frame_with_table_filter(frame, table_info)
            else:
                persons = pose_tracker.track_frame(frame)
            
            # プレイヤー分類器を更新
            if table_info and persons:
                player_classifier.update(persons, table_info, frame_count)
            
            # プレイヤーを分類
            if table_info:
                selected_ids, removed_ids = player_classifier.classify_players()
                player_ids = set(selected_ids)
                
                if removed_ids:
                    pose_tracker.remove_validated_track_ids(removed_ids)
            
            # 結果を描画
            display_frame = visualizer.draw_results(
                frame, table_info, persons, player_ids
            )
            display_frame = visualizer.draw_candidate_info(
                display_frame, player_ids
            )
            
            # フレーム情報を表示
            cv2.putText(
                display_frame,
                f"Frame: {frame_count}/{total_frames} (Processed: {processed_count})",
                (10, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )
            
            cv2.putText(
                display_frame,
                f"Detected: {len(persons)} persons, Players: {len(player_ids)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )
            
            # ビデオに保存
            video_writer.write(display_frame)
            
            # 次回のスキップフレーム用に保存
            last_persons = persons
    
    finally:
        pbar.close()
        cap.release()
        video_writer.release()
        
        # 最終結果を表示
        print(f"\n✓ 処理完了:")
        print(f"  処理フレーム数: {frame_count}")
        print(f"  実際に処理したフレーム数: {processed_count}")
        print(f"  検出されたプレイヤーID: {sorted(player_ids)}")
        print(f"  候補者数: {len(player_classifier.candidates)}")
        
        # 候補者詳細情報
        if player_classifier.candidates:
            print(f"\n=== 候補者詳細 ===")
            candidates = []
            for track_id, candidate in player_classifier.candidates.items():
                if candidate.total_frames >= player_classifier.min_tracking_frames:
                    score = player_classifier._calculate_player_score(candidate)
                    candidates.append((track_id, candidate, score))
            
            candidates.sort(key=lambda x: x[2], reverse=True)
            
            for track_id, candidate, score in candidates:
                is_player = track_id in player_ids
                print(f"\nID {track_id} {'[PLAYER]' if is_player else ''}:")
                print(f"  スコア: {score:.3f}")
                print(f"  フレーム数: {candidate.total_frames}")
                print(f"  総運動量: {candidate.total_movement:.1f}")
                print(f"  卓球台付近比率: {candidate.near_table_ratio:.1%}")
        
        print(f"\n出力ビデオ: {OUTPUT_VIDEO}")

# テストを実行
run_player_classification_test()

## 7. 結果の確認

In [ ]:
# 結果動画を表示
def show_video(video_path, width=800):
    """動画をNotebook内に表示"""
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width={width} controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

if Path(OUTPUT_VIDEO).exists():
    print("結果動画:")
    show_video(OUTPUT_VIDEO)
else:
    print("出力動画が見つかりません")

In [ ]:
# 結果動画をダウンロード
from google.colab import files
if Path(OUTPUT_VIDEO).exists():
    print("結果動画をダウンロードしています...")
    files.download(OUTPUT_VIDEO)
    print("✓ ダウンロードが完了しました")
else:
    print("出力動画が見つかりません")

## 8. GPU性能の確認

In [ ]:
# GPU使用状況を確認
!nvidia-smi

# PyTorchからGPU情報を確認
import torch
print(f"\nPyTorch GPU情報:")
print(f"  CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU名: {torch.cuda.get_device_name(0)}")
    print(f"  GPUメモリ: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")